# Intermediate NumPy

**Estimated time:** 45–60 minutes  
**Prerequisites:** Basic Python, familiarity with arrays and shape

## Learning Goals

| # | Topic |
|---|-------|
| 1 | Array creation patterns and dtypes |
| 2 | Advanced indexing — fancy, boolean, and `np.ix_` |
| 3 | Broadcasting rules |
| 4 | Vectorization and `np.vectorize` vs ufuncs |
| 5 | Linear algebra — dot products, matrix operations, solving systems |
| 6 | Random number generation (`np.random.default_rng`) |
| 7 | Structured arrays and `np.where` / `np.select` |
| 8 | Performance — views vs copies, memory layout |

---

### Quick Reference

```python
a.shape, a.ndim, a.dtype, a.size
np.arange(start, stop, step)
np.linspace(start, stop, n)
a[np.array([0,2,4])]               # fancy index
a[a > 0]                           # boolean index
a @ b                              # matrix multiply
np.linalg.solve(A, b)
rng = np.random.default_rng(seed)
```

In [ ]:
import numpy as np
print('numpy', np.__version__)

---
## Section 1 — Array Creation and Dtypes

NumPy arrays have a fixed dtype. Choosing the right dtype reduces memory and can speed up computation.

| dtype | bytes | range / precision |
|-------|-------|-------------------|
| `int8` | 1 | –128 to 127 |
| `int32` | 4 | –2B to 2B |
| `int64` | 8 | very large integers |
| `float32` | 4 | ~7 decimal digits |
| `float64` | 8 | ~15 decimal digits (default) |
| `bool` | 1 | True/False |

Creating arrays beyond `np.array()`:
- `np.zeros`, `np.ones`, `np.full`, `np.eye`
- `np.arange`, `np.linspace`, `np.logspace`
- `np.empty` — **uninitialized** (fastest, but values are garbage)

In [ ]:
# Different creation strategies
a_int = np.arange(1, 13, dtype=np.int32).reshape(3, 4)
a_float = np.linspace(0, 1, 9).reshape(3, 3)
identity = np.eye(4)
diagonal = np.diag([10, 20, 30, 40])

print('int32 array (3x4):')
print(a_int)
print('\nlinspace float64 (3x3):')
print(a_float.round(3))

In [ ]:
# Memory comparison
n = 1_000_000
arr64 = np.ones(n, dtype=np.float64)
arr32 = np.ones(n, dtype=np.float32)

print(f'float64: {arr64.nbytes / 1e6:.1f} MB')
print(f'float32: {arr32.nbytes / 1e6:.1f} MB')

In [ ]:
# Build a loss development triangle (upper-triangular structure)
# rows = accident years, cols = development ages
triangle = np.array([
    [1_000, 1_500, 1_700, 1_750],
    [  900, 1_350, 1_530,     0],
    [1_200, 1_800,     0,     0],
    [  800,     0,     0,     0],
], dtype=np.float64)

# Replace 0 with NaN for proper masking
triangle[triangle == 0] = np.nan
print(triangle)

In [ ]:
# EXERCISE 1:
# a) Create an array of 20 values log-spaced from 10^0 to 10^4
# b) Create a 5x5 array where entry [i,j] = i * j  (multiplication table)
#    Hint: use np.arange and broadcasting, or np.outer
# YOUR CODE HERE

---
## Section 2 — Advanced Indexing

NumPy has three indexing modes:

| Mode | Syntax | Returns copy or view? |
|------|--------|----------------------|
| Basic (slice) | `a[1:3, 0:2]` | **View** (no copy) |
| Boolean mask | `a[a > 5]` | **Copy** |
| Fancy (integer array) | `a[[0, 2, 4]]` | **Copy** |

`np.ix_` creates an open mesh for selecting sub-matrices with fancy indexing.

In [ ]:
mat = np.arange(25, dtype=float).reshape(5, 5)
print('Full matrix:')
print(mat)

# Basic slice — this is a VIEW
view = mat[1:3, 2:4]
print('\nSlice view [1:3, 2:4]:')
print(view)

# Modifying view changes original!
view[0, 0] = 999
print('\nmat after modifying view:')
print(mat)

In [ ]:
# Reset
mat = np.arange(25, dtype=float).reshape(5, 5)

# Boolean mask — select elements > 10 and set them to 0
mask = mat > 10
print('Mask shape:', mask.shape)
mat_clipped = mat.copy()
mat_clipped[mask] = 0
print(mat_clipped)

In [ ]:
mat = np.arange(25, dtype=float).reshape(5, 5)

# Fancy indexing — pick rows [0,2,4] and cols [1,3]
# np.ix_ creates the right broadcasting shape automatically
rows = np.array([0, 2, 4])
cols = np.array([1, 3])

sub = mat[np.ix_(rows, cols)]  # 3x2 sub-matrix
print('Sub-matrix (rows 0,2,4 x cols 1,3):')
print(sub)

# Compare: without np.ix_ you'd get diagonal pairs, not all combinations
print('\nWithout np.ix_ (element pairs, not sub-matrix):')
print(mat[[0, 2], [1, 3]])  # only 2 elements

In [ ]:
# Practical: extract the latest diagonal from the loss triangle
t = np.array([
    [1_000, 1_500, 1_700, 1_750],
    [  900, 1_350, 1_530,   np.nan],
    [1_200, 1_800,   np.nan, np.nan],
    [  800,   np.nan, np.nan, np.nan],
])

n = t.shape[0]
# Latest diagonal: for row i, take column (n-1-i)
diag_rows = np.arange(n)
diag_cols = n - 1 - diag_rows
latest_diagonal = t[diag_rows, diag_cols]
print('Latest diagonal:', latest_diagonal)

In [ ]:
# EXERCISE 2:
# Given the matrix below, use boolean indexing to:
# a) Find all values that are both > 5 and < 20
# b) Replace all negative values with 0
data = np.array([[ 3, -1, 12, 25],
                 [-5,  8,  0, 18],
                 [14, -3,  7, 22]])
# YOUR CODE HERE

---
## Section 3 — Broadcasting

Broadcasting lets NumPy operate on arrays with **different shapes** by virtually expanding dimensions.

### Rules (applied dimension by dimension, right-to-left)
1. If arrays have different number of dims, prepend 1s to the smaller shape.
2. Dimensions with size 1 are stretched to match the other.
3. If neither is 1 and sizes differ → error.

```
Shape (4, 3) + Shape (3,)   → (4, 3)   ✓  row broadcast
Shape (4, 1) + Shape (1, 3) → (4, 3)   ✓  both broadcast
Shape (4, 3) + Shape (4,)   → ERROR    ✗  last dims 3 ≠ 4
```

In [ ]:
# 1D broadcast: subtract column means from each column
mat = np.random.randint(10, 50, size=(5, 4)).astype(float)
col_means = mat.mean(axis=0)   # shape (4,)

print('Column means:', col_means)
centered = mat - col_means      # (5,4) - (4,) → (5,4)
print('Column means of centered (should be ~0):', centered.mean(axis=0).round(10))

In [ ]:
# Outer product via broadcasting — no np.outer needed
a = np.array([1, 2, 3, 4])      # shape (4,)
b = np.array([10, 20, 30])      # shape (3,)

outer = a[:, np.newaxis] * b[np.newaxis, :]  # (4,1) * (1,3) → (4,3)
print(outer)

In [ ]:
# Actuarial example: apply a different expense load to each row (accident year)
premiums = np.array([1_000_000, 1_200_000, 1_500_000, 1_800_000])  # shape (4,)
expense_loads = np.array([0.28, 0.30, 0.32])                       # shape (3,) — 3 scenarios

# We want net premiums for each (accident_year, scenario) combo
# premiums[:, np.newaxis] is (4,1), expense_loads is (3,) → result (4,3)
net_premium = premiums[:, np.newaxis] * (1 - expense_loads)
print('Net premium (rows=accident years, cols=scenarios):')
print(net_premium.astype(int))

In [ ]:
# EXERCISE 3:
# loss_triangle shape (4 accident years x 4 development ages):
loss = np.array([
    [500, 750, 850, 900],
    [600, 900, 990,   0],
    [700, 980,   0,   0],
    [550,   0,   0,   0],
], dtype=float)
loss[loss == 0] = np.nan

# Using broadcasting:
# a) Compute each cell as a % of that row's maximum (latest observed) value
#    Hint: row max = np.nanmax(loss, axis=1, keepdims=True)  → shape (4,1)
# b) Standardize each column (subtract col mean, divide by col std) — ignore NaN
# YOUR CODE HERE

---
## Section 4 — Vectorization: ufuncs vs np.vectorize

**Universal functions (ufuncs)** are C-level loops: `np.add`, `np.exp`, `np.log`, comparison ops, etc.

`np.vectorize` is a **convenience wrapper** — it still calls Python in a loop and is not truly fast. Use it only when no ufunc alternative exists.

| Approach | Speed | Use when |
|----------|-------|----------|
| ufunc / arithmetic | Fastest | Math on whole arrays |
| `np.where` / `np.select` | Fast | Conditional element-wise |
| `np.vectorize` | Slow (Python loop) | Last resort for custom scalar logic |
| Python `for` loop | Slowest | Never on large arrays |

In [ ]:
x = np.linspace(0.01, 10, 1_000_000)

# ufunc: log-normal distribution PDF — pure NumPy
mu, sigma = 1.5, 0.5
pdf = (1 / (x * sigma * np.sqrt(2 * np.pi))) * np.exp(-((np.log(x) - mu) ** 2) / (2 * sigma**2))
print('PDF computed for 1M points, max:', pdf.max().round(4))

In [ ]:
import math

small_x = np.linspace(0.01, 10, 100_000)

def scalar_pdf(x_val):
    return (1 / (x_val * sigma * math.sqrt(2 * math.pi))) * math.exp(-((math.log(x_val) - mu)**2) / (2 * sigma**2))

vec_pdf = np.vectorize(scalar_pdf)

print('np.vectorize:')
%timeit vec_pdf(small_x)

print('\nNumPy ufuncs:')
%timeit (1/(small_x*sigma*np.sqrt(2*np.pi)))*np.exp(-((np.log(small_x)-mu)**2)/(2*sigma**2))

In [ ]:
# np.select: multi-condition branching without Python loops
loss_ratios = np.array([0.40, 0.62, 0.75, 0.88, 1.10, 0.55])

conditions = [
    loss_ratios < 0.50,
    loss_ratios < 0.70,
    loss_ratios < 0.90,
]
choices = ['Excellent', 'Acceptable', 'Elevated']

rating = np.select(conditions, choices, default='Unacceptable')
print(list(zip(loss_ratios, rating)))

In [ ]:
# EXERCISE 4:
# Compute the present value of a stream of cash flows using vectorized ops
# PV_t = CF_t / (1 + r)^t
cash_flows = np.array([0, 500_000, 750_000, 900_000, 600_000, 400_000])  # t=0..5
r = 0.05
# a) Compute the PV of each cash flow (no loops)
# b) Compute the total NPV
# YOUR CODE HERE

---
## Section 5 — Linear Algebra

Key `np.linalg` functions:

| Function | Purpose |
|----------|---------|
| `a @ b` | Matrix multiply |
| `np.linalg.solve(A, b)` | Solve Ax = b |
| `np.linalg.inv(A)` | Matrix inverse (prefer solve over inv) |
| `np.linalg.det(A)` | Determinant |
| `np.linalg.eig(A)` | Eigenvalues and eigenvectors |
| `np.linalg.lstsq(A, b)` | Least-squares solution |

**Tip:** Avoid `np.linalg.inv` for solving systems — it's slower and less numerically stable than `solve`.

In [ ]:
# Matrix multiply: portfolio return = weights @ returns
weights = np.array([0.4, 0.35, 0.25])          # 3 asset classes
annual_returns = np.array([0.08, 0.05, 0.03])  # bonds, equities, cash

portfolio_return = weights @ annual_returns
print(f'Portfolio return: {portfolio_return:.2%}')

# Covariance matrix (3x3)
cov = np.array([
    [0.04, 0.01, 0.00],
    [0.01, 0.09, 0.01],
    [0.00, 0.01, 0.01],
])

# Portfolio variance = w^T * Cov * w
portfolio_var = weights @ cov @ weights
portfolio_vol = np.sqrt(portfolio_var)
print(f'Portfolio volatility: {portfolio_vol:.2%}')

In [ ]:
# Solve a system of equations — e.g. credibility weighting
# Suppose we have 3 unknowns (loadings) and 3 constraints
A = np.array([
    [1.0, 1.0, 1.0],  # loadings sum to 1
    [2.0, 1.0, 0.5],  # weighted mean constraint
    [1.0, 0.0, 2.0],  # variance constraint
])
b = np.array([1.0, 1.5, 2.0])

x = np.linalg.solve(A, b)
print('Solution x:', x.round(4))
print('Verify Ax = b:', np.allclose(A @ x, b))

In [ ]:
# Least squares: fit a line to noisy data
rng = np.random.default_rng(0)
t = np.arange(10)
y = 2.5 * t + 10 + rng.normal(0, 2, size=10)  # true slope=2.5, intercept=10

# Design matrix [t, 1]
A_ls = np.column_stack([t, np.ones(10)])
coeffs, residuals, rank, sv = np.linalg.lstsq(A_ls, y, rcond=None)
print(f'Fitted slope: {coeffs[0]:.3f}  intercept: {coeffs[1]:.3f}')

In [ ]:
# EXERCISE 5:
# Chain Ladder in NumPy:
# Given the cumulative loss triangle, compute volume-weighted LDFs for each column transition
# LDF(d -> d+1) = sum of column d+1 (observed) / sum of column d (same rows)
tri = np.array([
    [1000, 1500, 1700, 1750],
    [ 900, 1350, 1530,  np.nan],
    [1200, 1800,  np.nan, np.nan],
    [ 800,  np.nan, np.nan, np.nan],
])

# Hint: for transition d -> d+1, only use rows where both columns are observed (not NaN)
# YOUR CODE HERE — compute ldfs as a 1D array of length 3

---
## Section 6 — Random Number Generation

NumPy 1.17+ recommends `np.random.default_rng(seed)` over the legacy `np.random.seed()`. The new Generator API is reproducible, faster, and supports more distributions.

| Method | Distribution |
|--------|--------------|
| `rng.integers(low, high, size)` | Uniform integer |
| `rng.uniform(low, high, size)` | Uniform float |
| `rng.normal(mu, sigma, size)` | Gaussian |
| `rng.lognormal(mu, sigma, size)` | Log-normal |
| `rng.exponential(scale, size)` | Exponential |
| `rng.poisson(lam, size)` | Poisson |
| `rng.gamma(shape, scale, size)` | Gamma |
| `rng.choice(a, size, replace, p)` | Sampling |

In [ ]:
rng = np.random.default_rng(seed=42)

# Simulate 10,000 individual claim severities (log-normal)
claims = rng.lognormal(mean=np.log(25_000), sigma=1.2, size=10_000)
print(f'Mean:   ${claims.mean():,.0f}')
print(f'Median: ${np.median(claims):,.0f}')
print(f'95th %: ${np.percentile(claims, 95):,.0f}')
print(f'99th %: ${np.percentile(claims, 99):,.0f}')

In [ ]:
# Monte Carlo: estimate aggregate loss distribution
# Number of claims ~ Poisson(lam=50)
# Severity ~ LogNormal(mu=log(25000), sigma=1.2)

n_sims = 50_000
rng2 = np.random.default_rng(7)

claim_counts = rng2.poisson(lam=50, size=n_sims)

# Generate all severities at once, then split by count (efficient)
total_claims = claim_counts.sum()
all_severities = rng2.lognormal(np.log(25_000), 1.2, size=total_claims)

# np.split on cumulative counts
splits = np.cumsum(claim_counts)[:-1]
sim_losses = np.array([grp.sum() for grp in np.split(all_severities, splits)])

print('Aggregate loss distribution (50k simulations):')
for p in [50, 75, 90, 95, 99, 99.5]:
    print(f'  {p}th percentile: ${np.percentile(sim_losses, p):>15,.0f}')

In [ ]:
# EXERCISE 6:
# Bootstrap confidence interval for the mean claim severity
# Use the 'claims' array from above (10,000 severities)
# a) Draw 10,000 bootstrap samples of size 200 (with replacement)
# b) Compute the mean of each sample
# c) Report the 95% CI (2.5th and 97.5th percentiles of bootstrap means)
# Hint: rng.choice(claims, size=(10_000, 200), replace=True).mean(axis=1)
# YOUR CODE HERE

---
## Section 7 — np.where and np.select

These replace conditional logic over arrays without Python loops.

```python
np.where(condition, x, y)         # binary: if True → x, else → y
np.select([c1,c2,c3], [v1,v2,v3], default=v_else)  # multi-way
np.clip(a, a_min, a_max)          # cap values
np.nan_to_num(a, nan=0)           # replace NaN
```

In [ ]:
# Apply a reinsurance per-occurrence limit + retention
gross_losses = np.array([50_000, 120_000, 450_000, 1_200_000, 80_000, 2_500_000])
retention = 500_000

# Net loss = min(gross, retention)
net_losses = np.where(gross_losses > retention, retention, gross_losses)
# Reinsurance recovery = gross - net
ri_recovery = gross_losses - net_losses

for g, n, r in zip(gross_losses, net_losses, ri_recovery):
    print(f'Gross: {g:>10,}  Net: {n:>10,}  RI: {r:>10,}')

In [ ]:
# Working with NaN in triangles
tri = np.array([
    [1000., 1500., 1700., 1750.],
    [ 900., 1350., 1530.,   np.nan],
    [1200., 1800.,   np.nan, np.nan],
    [ 800.,   np.nan, np.nan, np.nan],
])

# Compute age-to-age factors only for observed transitions
# Shift: next = tri[:, 1:], prior = tri[:, :-1]
prior = tri[:, :-1]
nxt   = tri[:, 1:]

# Link ratios where both are observed
link_ratios = np.where(~np.isnan(nxt), nxt / prior, np.nan)
print('Link ratios:')
print(link_ratios.round(4))

# Volume-weighted LDFs
ldfs = np.nansum(nxt, axis=0) / np.nansum(prior, axis=0)
print('\nVolume-weighted LDFs:', ldfs.round(4))

In [ ]:
# EXERCISE 7:
# Using ldfs and tri from above:
# a) Compute cumulative development factors (CDFs) to ultimate
#    CDF[i] = product of ldfs[i:] (cumprod from right)
#    Hint: np.cumprod on the reversed array, then reverse back
#    Assume tail factor = 1.0
# b) Project each accident year's ultimate = latest_diagonal * CDF
# YOUR CODE HERE

---
## Section 8 — Views vs Copies and Memory Layout

Understanding views vs copies prevents subtle bugs and reduces memory allocation.

| Operation | View or Copy? |
|-----------|---------------|
| `a[1:3]` (basic slice) | View |
| `a[[0,2]]` (fancy index) | Copy |
| `a[a>0]` (boolean mask) | Copy |
| `a.reshape(...)` | View (usually) |
| `a.T` | View |
| `a.flatten()` | Copy |
| `a.ravel()` | View (when possible) |

**C-order (row-major)** is NumPy's default — rows are contiguous. Transposing gives Fortran order; iterating over columns of a C-order array is slower.

In [ ]:
# Check if an array owns its data
a = np.arange(12).reshape(3, 4)
view = a[0:2, :]      # slice → view
copy = a[[0, 1], :]   # fancy → copy

print('view.base is a:', view.base is a)  # True
print('copy.base is a:', copy.base is a)  # False
print('copy.base is None:', copy.base is None)  # True

# Safer: use .copy() explicitly when you don't want aliasing
safe_copy = a[0:2, :].copy()
print('safe_copy.base is a:', safe_copy.base is a)  # False

In [ ]:
# C-order (row-major) vs F-order (column-major) performance
big_c = np.random.rand(2000, 2000)                     # C-order default
big_f = np.asfortranarray(np.random.rand(2000, 2000))  # F-order

print('Summing along axis=1 (rows):')
%timeit big_c.sum(axis=1)    # contiguous read for C-order
%timeit big_f.sum(axis=1)    # strided read for F-order

print('\nSumming along axis=0 (cols):')
%timeit big_c.sum(axis=0)    # strided for C-order
%timeit big_f.sum(axis=0)    # contiguous for F-order

---
## Wrap-Up Cheat Sheet

| Topic | Key takeaway |
|-------|--------------|
| dtypes | Match dtype to data — `float32` halves memory vs `float64` |
| Indexing | Slices = views (fast, aliased); fancy/boolean = copies |
| Broadcasting | Align shapes right-to-left; size-1 dims stretch |
| Vectorization | Avoid Python loops — use ufuncs, `np.where`, `np.select` |
| Linear algebra | Prefer `solve` over `inv`; use `@` for matmul |
| RNG | Use `default_rng(seed)` for reproducibility |
| NaN | `np.nansum`, `np.nanmean` skip NaN; `np.isnan` for masking |
| Views | Modifying a view changes the original — use `.copy()` deliberately |